# Chapter 4: Your First LLM Workflow

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build a **Requirements Analyzer** -- an end-to-end workflow that takes raw requirements text, classifies them, identifies issues, and produces a structured quality report.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Completed Chapter 3 lab


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Define the Requirements Input

We start with a realistic set of raw requirements that contain common quality issues.


In [ ]:
# Sample raw requirements (intentionally imperfect)
raw_requirements = """
REQ-001: The system shall allow users to log in using their email and password.
REQ-002: The system should be fast and responsive.
REQ-003: Users must be able to upload documents in various formats.
REQ-004: The system shall send email notifications when an order is placed, 
          and also when it ships, and the notification should contain all relevant 
          information about the order.
REQ-005: The admin panel shall provide reporting capabilities.
REQ-006: The system shall encrypt all data at rest using AES-256 and in transit using TLS 1.3.
REQ-007: Users should be able to do things easily.
REQ-008: The search function shall return results within 2 seconds for queries 
          against up to 1 million records.
"""

print('Loaded', raw_requirements.count('REQ-'), 'requirements for analysis')

## 2. Step 1 -- Classify Requirements

The first step in our workflow classifies each requirement by type.


In [ ]:
# Step 1: Classify requirements
classify_prompt = f"""Classify each requirement below into one of these categories:
- Functional
- Non-Functional (Performance)
- Non-Functional (Security)
- Non-Functional (Usability)
- Non-Functional (Other)

Return a JSON array with objects containing: id, text (first 50 chars), category.

Requirements:
{raw_requirements}

Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': classify_prompt}],
    temperature=0.1
)
classified = json.loads(response.choices[0].message.content)
for req in classified:
    print(f"{req['id']}: {req['category']}")

## 3. Step 2 -- Quality Analysis

Analyze each requirement for common quality issues.


In [ ]:
# Step 2: Analyze quality
quality_prompt = f"""Analyze each requirement for quality issues. Check for:
1. Ambiguity (vague terms like 'fast', 'various', 'easily')
2. Completeness (missing details or conditions)
3. Testability (can it be objectively verified?)
4. Atomicity (does it contain multiple requirements?)
5. Consistency (conflicts with other requirements)

For each requirement, return a JSON object with:
- id: requirement ID
- quality_score: 1-10
- issues: list of {{"type": "...", "description": "...", "severity": "high|medium|low"}}
- improved_version: rewritten requirement that fixes the issues

Requirements:
{raw_requirements}

Return a JSON array. Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': quality_prompt}],
    temperature=0.2
)
quality_results = json.loads(response.choices[0].message.content)

for req in quality_results:
    print(f"\n{req['id']} - Score: {req['quality_score']}/10")
    for issue in req['issues']:
        print(f"  [{issue['severity'].upper()}] {issue['type']}: {issue['description']}")

In [ ]:
# Step 3: Generate summary report
report_prompt = f"""Based on this requirements quality analysis, generate an executive summary report.

Classification: {json.dumps(classified)}
Quality Analysis: {json.dumps(quality_results)}

Include:
1. Overall quality score (average)
2. Distribution by category
3. Top 3 critical issues
4. Recommendations for the team
5. Improved versions of the worst-scoring requirements"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': report_prompt}],
    temperature=0.4
)
print(response.choices[0].message.content)

## Exercises
1. Add a Step 4 that generates test cases for the highest-quality requirements.
2. Modify the workflow to accept requirements from a CSV or Excel file.
3. Add traceability -- link each requirement to its classification and quality issues in one final JSON output.
4. Create a reusable `RequirementsAnalyzer` class that encapsulates this entire workflow.
